# A4 · QC del cubo (M1–M5)

**Spec:** [`docs/spec_A4_codex_cube_qc.md`](../docs/spec_A4_codex_cube_qc.md)  |  **Bloque:** A · Reducción  |  **Run por defecto:** `ROXs12b_realigned`

Métricas de calidad del cubo: solución en λ (M1/M2), flujo absoluto (M3), STAT (M5).

| | |
|---|---|
| **Entrada** | `cube_telcorr.fits`, SKY_SPECTRUM, Gaia DR3 |
| **Salida (QC/productos)** | `stages/stage00q_qc.json` |
| **Consume aguas abajo** | D2/E1 (usan σ empírico), E3 (flujo) |


## Cómo ejecutar de forma independiente

> ⚠️ **Etapa no re-ejecutable desde raw en este repo.** En la poda WP-10 se borraron los intermedios regenerables (`muse_scibasic`, `muse_scipost`, …). Se conservaron los productos finales y todo el QC. Este notebook **audita** el producto/QC existente y documenta el comando histórico.

Comando histórico (referencia, requiere los raw + `esorex`):

```bash
conda activate MUSE
# M1/M2 (LSF) desde el airglow cacheado:
python -m musepipe.qc.cube_qc m1m2-sky --sky-spectrum <SKY_SPECTRUM...> --qc-output <...>
# M3 (flujo absoluto vs Gaia RP, con growth-curve + truncación):
python -m musepipe.qc.cube_qc m3-flux --cube <cube_telcorr.fits> --run-id $RUN \
    --aperture-correction growth_curve --truncation-correction --qc-output <...>
```


In [ ]:
import os, sys
# Añade notebooks/ (para _nbcommon) y la RAÍZ del repo (para importar musepipe),
# funcione el cwd en notebooks/ o en la raíz del repo.
_here = os.getcwd()
if os.path.basename(_here) != 'notebooks' and os.path.isdir(os.path.join(_here, 'notebooks')):
    _here = os.path.join(_here, 'notebooks')
for _p in (_here, os.path.dirname(_here)):
    if _p not in sys.path:
        sys.path.insert(0, _p)
import _nbcommon as nb
_root = str(nb.project_root())
if _root not in sys.path:
    sys.path.insert(0, _root)   # asegura 'import musepipe'
RUN_ID = nb.resolve_run_id(None)
print('run  =', RUN_ID)
print('root =', _root)
print('dir  =', nb.run_dir(RUN_ID))


## Auditar

Etapa de solo-auditoría: se carga el producto/QC más abajo.


## QC / resultados


In [ ]:
qc = nb.load_qc('stages/stage00q_qc.json', RUN_ID)
nb.show(qc, keys=['m1', 'm2', 'lsf', 'm3', 'flux_factor', 'm5', 'factor_spaxel'], title='A4')


## Decisiones y notas
- **M3 CERRADO (GREEN)**: flujo absoluto validado vs Gaia DR3 RP, factor 0.973 (~3%) tras growth-curve + truncación de cola.
- **M5 STAT en ROJO (inherente)**: el STAT subestima el ruido ~4–6× por covarianza del remuestreo → σ SIEMPRE empírico, control=objeto. · [`docs/noise_model.md`](../docs/noise_model.md)
- **M2 LSF@Hα = 2.383 Å medido** del airglow (NFM más angosta que el nominal 2.6); usado en E1/E3.


## Checks


In [ ]:
nb.show(qc, keys=['m3','flux_factor','m5','lsf'])
